# 11 — Serialization, Reload, Inference, Latency, and Throughput (Manual NumPy)


> **Learning contract.** Every code cell is preceded by an explanation of what the code does, why the operation exists mathematically, what tensor/array shapes are expected, and what production or business failure it prevents. Run the notebooks in numerical order in a fresh Conda environment.


Training and inference are different workloads. Inference must reproduce preprocessing, load a versioned artifact, accept a stable input contract, produce deterministic output semantics, and satisfy latency/throughput constraints.

**Input contract:** one grayscale 28×28 image or a batch, converted to `float32`, flattened to 784, normalized to `[0,1]`. **Output contract:** 10 logits/probabilities plus predicted class; business systems may add confidence thresholds or abstention.


## Code walkthrough — reload the artifact and verify deterministic predictions
This cell intentionally starts from disk. Persisted artifacts are the unit of deployment. If a reloaded model does not reproduce the expected prediction path, packaging or serialization is broken.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
MNIST_PATH = DATA_DIR / "mnist.npz"
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"


def load_official_mnist():
    if not MNIST_PATH.exists():
        print("Downloading official MNIST archive to", MNIST_PATH)
        urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)
    with np.load(MNIST_PATH) as data:
        return (data["x_train"], data["y_train"], data["x_test"], data["y_test"])


def balanced_subset(x, y, per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    selected = []
    for cls in range(10):
        candidates = np.flatnonzero(y == cls)
        selected.extend(rng.choice(candidates, size=per_class, replace=False))
    selected = np.asarray(selected)
    rng.shuffle(selected)
    return (x[selected], y[selected])


def prepare_splits():
    x_train_raw, y_train_raw, x_test_raw, y_test_raw = load_official_mnist()
    x_dev, y_dev = balanced_subset(x_train_raw, y_train_raw, per_class=600)
    x_test, y_test = balanced_subset(
        x_test_raw, y_test_raw, per_class=100, seed=SEED + 1
    )
    x_train, x_val, y_train, y_val = train_test_split(
        x_dev, y_dev, test_size=1000, random_state=SEED, stratify=y_dev
    )

    def transform(x):
        return x.reshape(len(x), -1).astype("float32") / 255.0

    return (
        transform(x_train),
        y_train,
        transform(x_val),
        y_val,
        transform(x_test),
        y_test,
    )


rng = np.random.default_rng(SEED)
W1 = rng.normal(0, np.sqrt(2 / 784), size=(784, 64)).astype("float32")
b1 = np.zeros((1, 64), dtype="float32")
W2 = rng.normal(0, np.sqrt(2 / 64), size=(64, 10)).astype("float32")
b2 = np.zeros((1, 10), dtype="float32")


def relu(z):
    return np.maximum(z, 0)


def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def forward(X):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    logits = a1 @ W2 + b2
    return (z1, a1, logits)


X_train, y_train, X_val, y_val, X_test, y_test = prepare_splits()
a = np.load(ARTIFACT_DIR / "mnist_manual_mlp.npz")
W1, b1, W2, b2 = [a[k] for k in ["W1", "b1", "W2", "b2"]]
z = np.maximum(X_test @ W1 + b1, 0) @ W2 + b2
e = np.exp(z - z.max(1, keepdims=True))
probs = e / e.sum(1, keepdims=True)
print("first five true", y_test[:5])
print("first five predicted", probs[:5].argmax(1))
print("first five confidence", np.round(probs[:5].max(1), 3))

first five true [0 3 0 8 2]
first five predicted [0 3 0 8 2]
first five confidence [0.948 0.357 0.693 0.672 0.999]


## Code walkthrough — measure single-item latency and batched throughput
Microbenchmarks are noisy, but they teach an important systems principle: batching often improves throughput by amortizing framework and matrix-operation overhead. Production benchmarking should include warm-up, percentile latency, hardware specification and realistic concurrency.


In [2]:
import time


def infer_np(X):
    h = np.maximum(X @ W1 + b1, 0)
    logits = h @ W2 + b2
    e = np.exp(logits - logits.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


for _ in range(20):
    infer_np(X_test[:256])
start = time.perf_counter()
for _ in range(200):
    infer_np(X_test[:1])
single_ms = (time.perf_counter() - start) / 200 * 1000
start = time.perf_counter()
for _ in range(100):
    infer_np(X_test[:256])
batch_sec = time.perf_counter() - start
throughput = 100 * 256 / batch_sec
print(f"actual model single-item latency ~{single_ms:.3f} ms")
print(f"actual model batch throughput ~{throughput:,.0f} images/sec")

actual model single-item latency ~0.017 ms
actual model batch throughput ~1,100,148 images/sec


## Production implication
Model latency is only one component of request latency. Image decoding, feature lookup, network hops, serialization, model runtime and downstream policy all contribute. Monitor end-to-end service percentiles, not only notebook timing.
